In [1]:
import numpy as np
from dataclasses import dataclass
from typing import Callable, Optional, Tuple
from scipy.linalg import expm
from scipy.integrate import quad


# -------------------------
# 1) Non-uniform grid (Section 4)  :contentReference[oaicite:3]{index=3}
# -------------------------

def generate_subgrid(a: float, s: float, b: float, M: int, g1: float, g2: float) -> np.ndarray:
    """
    GenerateSubGrid(a,s,b,M,g1,g2) from the paper (Section 4).
    Requires M even.
    """
    if M % 2 != 0:
        raise ValueError("M must be even.")
    if not (a < s < b):
        raise ValueError("Need a < s < b.")

    c1 = np.arcsinh((a - s) / g1)
    c2 = np.arcsinh((b - s) / g2)

    half = M // 2
    # lower part: k=1..half, x1=a, x_half=s
    k = np.arange(1, half + 1)
    x_lower = s + g1 * np.sinh(c1 * (1 - (k - 1) / (half - 1)))

    # upper part: k=1..half, x_{half+1}..x_M, x_M=b
    k = np.arange(1, half + 1)
    x_upper = s + g2 * np.sinh(c2 * (2 * k / M))

    return np.concatenate([x_lower, x_upper])


def generate_grid(
    S0: float,
    ell: float,
    u: float,
    x_min: float,
    x_max: float,
    N1: int,
    N2: int,
    N3: int,
    d1_minus: float, d1_plus: float,
    d2_minus: float, d2_plus: float,
    d3_minus: float, d3_plus: float,
) -> np.ndarray:
    """
    Concatenate 3 subgrids as in Section 4:
      [x_min, ell], [ (S0+ell)/2, S0, (u+S0)/2 ], [u, x_max]
    and ensure ell,S0,u are on the grid.
    """
    if not (x_min < ell < S0 < u < x_max):
        raise ValueError("Need x_min < ell < S0 < u < x_max.")
    N = N1 + N2 + N3
    if N1 % 2 or N2 % 2 or N3 % 2:
        raise ValueError("N1,N2,N3 must be even.")

    # subgrid 1
    G1 = generate_subgrid(x_min, ell, 0.5 * (S0 + ell), N1, d1_minus, d1_plus)
    # subgrid 2
    G2 = generate_subgrid(0.5 * (S0 + ell), S0, 0.5 * (u + S0), N2, d2_minus, d2_plus)
    # subgrid 3
    G3 = generate_subgrid(0.5 * (u + S0), u, x_max, N3, d3_minus, d3_plus)

    # concatenate, remove near-duplicates, sort
    G = np.unique(np.concatenate([G1, G2, G3]))
    G.sort()

    # enforce exact inclusion (numerical hygiene)
    for z in [ell, S0, u]:
        if np.all(np.abs(G - z) > 1e-12):
            G = np.sort(np.append(G, z))

    return G


# -------------------------
# 2) Diffusion generator (tri-diagonal)  :contentReference[oaicite:4]{index=4}
# -------------------------

def build_diffusion_generator(
    G: np.ndarray,
    r: float,
    d: float,
    sigma: Callable[[float], float],
) -> np.ndarray:
    """
    Tri-diagonal Λ_D matching local drift (r-d)x and local variance sigma(x)^2 x^2
    using standard non-uniform finite differences (birth-death style).
    Boundary rows are set to 0 (absorbing / “stopped at boundary” convention).
    """
    n = len(G)
    Lam = np.zeros((n, n), dtype=float)
    gamma = r - d

    for i in range(1, n - 1):
        x = G[i]
        hm = x - G[i - 1]
        hp = G[i + 1] - x
        if hm <= 0 or hp <= 0:
            raise ValueError("Grid must be strictly increasing.")

        sig2 = (sigma(x) ** 2) * (x ** 2)

        # second derivative coeffs on nonuniform grid:
        # f''(xi) ≈ 2 * ( (f_{i+1}-f_i)/hp - (f_i-f_{i-1})/hm ) / (hp+hm)
        a = 2.0 / (hm * (hm + hp))   # multiplies f_{i-1}
        c = 2.0 / (hp * (hm + hp))   # multiplies f_{i+1}
        b = -(a + c)                 # multiplies f_i

        # first derivative: upwind based on sign of drift
        drift = gamma * x
        if drift >= 0:
            # backward diff: f'(xi) ≈ (f_i - f_{i-1})/hm
            d_im1 = -drift / hm
            d_i   =  drift / hm
            d_ip1 = 0.0
        else:
            # forward diff: f'(xi) ≈ (f_{i+1}-f_i)/hp
            d_im1 = 0.0
            d_i   = -drift / hp
            d_ip1 =  drift / hp

        # combine: (sig2/2) f'' + drift f'
        Lam[i, i - 1] += 0.5 * sig2 * a + d_im1
        Lam[i, i]     += 0.5 * sig2 * b + d_i
        Lam[i, i + 1] += 0.5 * sig2 * c + d_ip1

    # enforce generator properties: off-diagonals >= 0, rows sum to 0
    # (small negative off-diagonals can happen from discretisation; clip with caution)
    for i in range(1, n - 1):
        for j in [i - 1, i + 1]:
            if Lam[i, j] < 0:
                Lam[i, j] = 0.0
        Lam[i, i] = -np.sum(Lam[i, :]) + Lam[i, i]  # adjust (keeps diagonal negative)

        # final enforce exact zero row sum
        Lam[i, i] = -np.sum(Lam[i, :]) + Lam[i, i]

    return Lam


# -------------------------
# 3) Jump generator by bin-integrating ν(x,dy)  :contentReference[oaicite:5]{index=5}
# -------------------------

def build_jump_generator_from_levy_measure(
    G: np.ndarray,
    nu: Callable[[float, float], float],
    y_support: Tuple[float, float] = (-1.0, np.inf),
    quad_epsabs: float = 1e-10,
    quad_epsrel: float = 1e-8,
) -> np.ndarray:
    """
    Discretise jump measure ν(x,dy) on relative jump sizes y = z/x - 1
    by integrating ν(x,dy) over bins induced by the grid (Section 4.1).
    This is the generic (slower but clear) version.

    nu(x,y): density of ν(x,dy) w.r.t. dy on (-1,∞).
    """
    n = len(G)
    LamJ = np.zeros((n, n), dtype=float)

    # boundary rows = 0 (paper’s convention in many experiments)
    for i in range(1, n - 1):
        x = G[i]

        # relative jump sizes to every grid point
        y = (G / x) - 1.0  # y[j] corresponds to jumping from x to G[j]
        order = np.argsort(y)
        y_sorted = y[order]
        j_sorted = order

        # bin edges alpha(y_i) between consecutive relative jump sizes
        # midpoints are a simple choice (paper suggests midpoint as “natural”) :contentReference[oaicite:6]{index=6}
        edges = np.empty(n + 1)
        edges[0] = y_support[0]
        edges[-1] = y_support[1]
        edges[1:-1] = 0.5 * (y_sorted[:-1] + y_sorted[1:])

        # integrate density over each bin and assign intensity to the corresponding destination state
        for k in range(n):
            y_lo, y_hi = edges[k], edges[k + 1]
            # skip the "no-jump" destination (y≈0 => z≈x): you still can allow it, but it just folds into diagonal
            dest = j_sorted[k]
            if dest == i:
                continue

            # integrate ν(x,dy) over (y_lo, y_hi)
            val, _ = quad(lambda yy: nu(x, yy), y_lo, y_hi, epsabs=quad_epsabs, epsrel=quad_epsrel, limit=200)
            LamJ[i, dest] = max(val, 0.0)

        LamJ[i, i] = -np.sum(LamJ[i, :])

    return LamJ


# -------------------------
# 4) Barrier matrices and pricing (Theorem 1)  :contentReference[oaicite:7]{index=7}
# -------------------------

def make_hat_and_tilde(Lam: np.ndarray, G: np.ndarray, ell: float, u: float, r: float):
    """
    Build Λ̂ on Ĝ = {x in G: ell < x < u} and Λ̃_r on full G as in (3.6)-(3.7). :contentReference[oaicite:8]{index=8}
    """
    inside = (G > ell) & (G < u)
    idx_in = np.where(inside)[0]
    idx_out = np.where(~inside)[0]

    Lam_hat = Lam[np.ix_(idx_in, idx_in)].copy()

    # Λ̃_r: zero rows outside, subtract r on diagonal inside
    Lam_tilde = Lam.copy()
    Lam_tilde[idx_out, :] = 0.0
    Lam_tilde[idx_in, idx_in] -= r

    return idx_in, idx_out, Lam_hat, Lam_tilde


def price_knock_out(
    Lam: np.ndarray,
    G: np.ndarray,
    ell: float,
    u: float,
    T: float,
    payoff: Callable[[np.ndarray], np.ndarray],
) -> np.ndarray:
    """
    Price of a continuously monitored knock-out with payoff g(X_T) 1_{tau>T},
    using (3.9): exp(T Λ̂) g restricted to Ĝ. :contentReference[oaicite:9]{index=9}

    Returns a vector of prices on the *full* grid (0 outside Ĝ).
    """
    idx_in, _, Lam_hat, _ = make_hat_and_tilde(Lam, G, ell, u, r=0.0)
    g_full = payoff(G)
    g_in = g_full[idx_in]

    P_in = expm(T * Lam_hat) @ g_in
    out = np.zeros_like(G, dtype=float)
    out[idx_in] = P_in
    return out


def price_barrier_with_rebate(
    Lam: np.ndarray,
    G: np.ndarray,
    ell: float,
    u: float,
    T: float,
    r: float,
    f_at_stop: Callable[[np.ndarray], np.ndarray],
) -> np.ndarray:
    """
    General stopped-discounted value E[e^{-r(T∧tau)} f(X_{T∧tau})] via (3.8) with Λ̃_r. :contentReference[oaicite:10]{index=10}
    This covers rebates/absorbed contracts if you encode f correctly on Ĝ vs Ĝ^c.
    """
    _, _, _, Lam_tilde = make_hat_and_tilde(Lam, G, ell, u, r=r)
    f = f_at_stop(G)
    return expm(T * Lam_tilde) @ f


# -------------------------
# 5) Minimal example: GBM double knock-out call (no jumps)
# -------------------------

def payoff_call(K: float) -> Callable[[np.ndarray], np.ndarray]:
    return lambda x: np.maximum(x - K, 0.0)


if __name__ == "__main__":
    # Model: GBM under risk-neutral drift r-d, constant vol
    S0 = 100.0
    r, d = 0.02, 0.0
    sig0 = 0.2
    sigma = lambda x: sig0

    # Barriers and maturity
    ell, u = 80.0, 120.0
    T = 1.0
    K = 100.0

    # Grid params (similar spirit to the paper’s examples)
    G = generate_grid(
        S0=S0, ell=ell, u=u,
        x_min=50.0, x_max=150.0,
        N1=60, N2=80, N3=60,
        d1_minus=100, d1_plus=1,
        d2_minus=10,  d2_plus=10,
        d3_minus=1,   d3_plus=100,
    )

    LamD = build_diffusion_generator(G, r=r, d=d, sigma=sigma)
    Lam = LamD  # GBM => no jumps

    # Knock-out call price surface on the grid
    prices = price_knock_out(Lam, G, ell=ell, u=u, T=T, payoff=payoff_call(K))

    # Interpolate to S0 if S0 isn't exactly on grid (we forced it on grid, but still)
    i0 = np.argmin(np.abs(G - S0))
    print("Closest grid point:", G[i0], "KO call price:", prices[i0])

Closest grid point: 100.0 KO call price: 1.1050830544691763
